In [76]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [77]:
import pandas as pd
import numpy as np
import unicodedata
import re

# Hate

In [78]:
hate_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Raw/telugu_hate_dataset_20k.csv")

print("Original Shape:", hate_df.shape)
hate_df.head()

Original Shape: (20000, 2)


,text,label
0,మనసు శాంతిగా ఉంచుకోండి.,NEUTRAL
1,వాళ్లు పనికిరారు.,HATE
2,మీ అభిప్రాయం నాకు ముఖ్యమైనది.,NEUTRAL
3,మీ అభిప్రాయం నాకు ముఖ్యమైనది.,NEUTRAL
4,ఆ కమ్యూనిటీపై నమ్మకం లేదు.,HATE


In [ ]:
hate_df = hate_df.rename(columns={
    "text": "text",
    "label": "hate_label"
})

In [80]:
hate_df = hate_df.dropna(subset=["text", "hate_label"])

In [81]:
def normalize_text(text):
    return unicodedata.normalize("NFKC", str(text))

hate_df["text"] = hate_df["text"].apply(normalize_text)

In [82]:
def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

hate_df["text"] = hate_df["text"].apply(clean_text)

In [83]:
hate_label_map = {
    "Hate": 0,
    "Offensive": 1,
    "Neutral": 2
}

hate_df["hate_label"] = hate_df["hate_label"].astype(str).str.strip()
hate_df["hate_label"] = hate_df["hate_label"].map(hate_label_map)

print("Unique Hate Labels:", hate_df["hate_label"].unique())

Unique Hate Labels: [nan]


In [84]:
hate_df.to_csv("//content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_telugu_hate.csv", index=False)

# Emotion

In [85]:
emotion_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Raw/telugu_emotion_40k.csv")
emotion_df.shape

print(emotion_df.head())

                                                text  label
0  నిజంగా నాకు బాగా కోపం వస్తోంది ఈ పరిస్థితిలో. (0)  anger
1    కొన్నిసార్లు నాకు చికాకు కలుగుతోంది ఈ రోజు. (1)  anger
2         నిజంగా నాకు బాగా కోపం వస్తోంది ఈ రోజు. (2)  anger
3      నాకు బాగా కోపం వస్తోంది అలా అనిపిస్తోంది. (3)  anger
4  నిజంగా నాకు చికాకు కలుగుతోంది అలా అనిపిస్తోంది...  anger


In [86]:
emotion_df = emotion_df.rename(columns={
    "text":"text",
    "label":"emotion_label"
})

In [87]:
emotion_df = emotion_df.dropna(subset=["text", "emotion_label"])

In [88]:
emotion_df["text"] = emotion_df["text"].apply(normalize_text)
emotion_df["text"] = emotion_df["text"].apply(clean_text)

In [89]:
emotion_df["emotion_label"] = (
    emotion_df["emotion_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [90]:
emotion_label_map = {
    0: "Anger",
    1: "Hate",
    2: "Sadness",
    3: "Fear",
    4: "Neutral"
}

emotion_df["emotion_label"] = emotion_df["emotion_label"].map(emotion_label_map)

print("Unique Emotion Labels:", emotion_df["emotion_label"].unique())

Unique Emotion Labels: [nan]


In [91]:
emotion_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_telugu_emotion.csv", index=False)

## Sentiment

In [92]:
sentiment_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Raw/telugu_sentiment_dataset_20k.csv")
sentiment_df.shape
print(sentiment_df.head())

                             text     label
0   ఇది పూర్తిగా మూర్ఖపు నిర్ణయం.  negative
1         మనసు శాంతిగా ఉంచుకోండి.   neutral
2  చదువు మన జీవితాన్ని మార్చగలదు.   neutral
3  చదువు మన జీవితాన్ని మార్చగలదు.  positive
4         మనసు శాంతిగా ఉంచుకోండి.  positive


In [93]:
sentiment_df = sentiment_df.rename(columns={
    "text": "text",
    "label": "sentiment_label"
})

In [94]:
sentiment_df = sentiment_df.dropna(subset=["text", "sentiment_label"])

In [95]:
sentiment_df["text"] = sentiment_df["text"].apply(normalize_text)
sentiment_df["text"] = sentiment_df["text"].apply(clean_text)

In [96]:
sentiment_df["sentiment_label"] = (
    sentiment_df["sentiment_label"]
    .astype(str)
    .str.strip()
)

In [97]:
sentiment_label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}



sentiment_df["sentiment_label"] = sentiment_df["sentiment_label"].map(sentiment_label_map)

print("Unique Sentiment Labels:", sentiment_df["sentiment_label"].unique())

Unique Sentiment Labels: [0 1 2]


In [98]:
sentiment_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_telugu_sentiment.csv", index=False)

# **MULTI-TASK MERGE**

In [99]:
hate_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_telugu_hate.csv")
emotion_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_telugu_emotion.csv")
sentiment_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_telugu_sentiment.csv")

hate_df.shape, emotion_df.shape, sentiment_df.shape
#

((20000, 2), (40000, 2), (20000, 2))

In [100]:
hate_df["emotion_label"] = -100
hate_df["sentiment_label"] = -100

emotion_df["hate_label"] = -100
emotion_df["sentiment_label"] = -100

sentiment_df["hate_label"] = -100
sentiment_df["emotion_label"] = -100

In [101]:
hate_df = hate_df[["text", "hate_label", "emotion_label", "sentiment_label"]]
emotion_df = emotion_df[["text", "hate_label", "emotion_label", "sentiment_label"]]
sentiment_df = sentiment_df[["text", "hate_label", "emotion_label", "sentiment_label"]]

In [102]:
multi_df = pd.concat([hate_df, emotion_df, sentiment_df], axis=0)

multi_df = multi_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final Telugu Multi-task Shape:", multi_df.shape)

Final Telugu Multi-task Shape: (80000, 4)


In [103]:
multi_df.head()

,text,hate_label,emotion_label,sentiment_label
0,ఈ రోజు నాకు శాంతంగా ఉంది ఈ రోజు. (3044),-100.0,NaN,-100
1,కొన్నిసార్లు నాకు అన్నీ బాగానే ఉన్నాయి అలా అని...,-100.0,NaN,-100
2,నువ్వు ఏం తెలుసు?,-100.0,-100.0,0
3,చదువు మన జీవితాన్ని మార్చగలదు.,-100.0,-100.0,2
4,నిజంగా నాకు ఏమీ ప్రత్యేకం లేదు ఈ రోజు. (2645),-100.0,NaN,-100


In [104]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    multi_df,
    test_size=0.2,
    random_state=42
)

train_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/telugu_multitask_train.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/telugu_multitask_test.csv", index=False)

print("Telugu Multi-task Dataset Ready.")

Telugu Multi-task Dataset Ready.


In [105]:
train_df["hate_label"].value_counts()
train_df["emotion_label"].value_counts()
train_df["sentiment_label"].value_counts()

,count
sentiment_label,
-100,48008
2,5351
0,5322
1,5319


# Traning Phase

In [106]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report

In [107]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [108]:
train_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/telugu_multitask_train.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/telugu_multitask_test.csv")

print(train_df.shape, test_df.shape)

(64000, 4) (16000, 4)


In [109]:
print("NaNs in hate:", train_df["hate_label"].isnull().sum())
print("NaNs in emotion:", train_df["emotion_label"].isnull().sum())
print("NaNs in sentiment:", train_df["sentiment_label"].isnull().sum())

NaNs in hate: 16018
NaNs in emotion: 31990
NaNs in sentiment: 0


In [110]:
multi_df["hate_label"] = multi_df["hate_label"].fillna(-100)
multi_df["emotion_label"] = multi_df["emotion_label"].fillna(-100)
multi_df["sentiment_label"] = multi_df["sentiment_label"].fillna(-100)

In [111]:
multi_df = multi_df.astype({
    "hate_label": "int64",
    "emotion_label": "int64",
    "sentiment_label": "int64"
})

In [112]:
train_df = train_df.fillna(-100)

train_df["hate_label"] = train_df["hate_label"].astype(int)
train_df["emotion_label"] = train_df["emotion_label"].astype(int)
train_df["sentiment_label"] = train_df["sentiment_label"].astype(int)

In [ ]:
HF_TOKEN = "your hugging face token"

In [114]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "ai4bharat/indic-bert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
base_model = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN)

print("IndicBERT loaded successfully!")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


IndicBERT loaded successfully!


In [115]:
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.hate_labels = df["hate_label"].tolist()
        self.emotion_labels = df["emotion_label"].tolist()
        self.sentiment_labels = df["sentiment_label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "hate_label": torch.tensor(self.hate_labels[idx], dtype=torch.long),
            "emotion_label": torch.tensor(self.emotion_labels[idx], dtype=torch.long),
            "sentiment_label": torch.tensor(self.sentiment_labels[idx], dtype=torch.long)
        }

In [116]:
train_dataset = MultiTaskDataset(train_df, tokenizer)
test_dataset  = MultiTaskDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [117]:
class MultiTaskIndicBERT(nn.Module):
    def __init__(self, encoder, hate_classes, emotion_classes, sentiment_classes):
        super().__init__()

        self.encoder = encoder
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.3)

        self.hate_head = nn.Linear(hidden_size, hate_classes)
        self.emotion_head = nn.Linear(hidden_size, emotion_classes)
        self.sentiment_head = nn.Linear(hidden_size, sentiment_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = torch.mean(outputs.last_hidden_state, dim=1)
        pooled = self.dropout(pooled)

        hate_logits = self.hate_head(pooled)
        emotion_logits = self.emotion_head(pooled)
        sentiment_logits = self.sentiment_head(pooled)

        return hate_logits, emotion_logits, sentiment_logits

In [118]:
hate_classes = 3
emotion_classes = 5
sentiment_classes = 3

model = MultiTaskIndicBERT(
    base_model,
    hate_classes,
    emotion_classes,
    sentiment_classes
)

model.to(device)

MultiTaskIndicBERT(
  (encoder): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(200000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertAttention(
                (attention_dropout): Dropout(p=0, inplace=False)
                (output_dropout): Dropout(p=0, inplace=False)
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Lin

In [119]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 2
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [120]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        hate_labels = batch["hate_label"].to(device)
        emotion_labels = batch["emotion_label"].to(device)
        sentiment_labels = batch["sentiment_label"].to(device)

        optimizer.zero_grad()

        hate_logits, emotion_logits, sentiment_logits = model(input_ids, attention_mask)

        loss = 0

        if (hate_labels != -100).any():
            loss += criterion(hate_logits, hate_labels)

        if (emotion_labels != -100).any():
            loss += criterion(emotion_logits, emotion_labels)

        if (sentiment_labels != -100).any():
            loss += criterion(sentiment_logits, sentiment_labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

KeyboardInterrupt: 

In [121]:
hate_label_map = {
    0: "Hate",
    1: "Offensive",
    2: "Neutral"
}

emotion_label_map = {
    0: "Anger",
    1: "Hate",
    2: "Sadness",
    3: "Fear",
    4: "Neutral"
}

sentiment_label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

In [122]:
import torch
import torch.nn.functional as F

# -------------------------------
# LABEL MAPS (5 EMOTIONS)
# -------------------------------

hate_label_map = {
    0: "Hate",
    1: "Offensive",
    2: "Neutral"
}

emotion_label_map = {
    0: "anger",
    1: "hate",
    2: "sadness",
    3: "fear",
    4: "neutral"
}

sentiment_label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

# -------------------------------
# PREDICTION FUNCTION
# -------------------------------

def telugu_predict(text):

    model.eval()

    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        hate_logits, emotion_logits, sentiment_logits = model(input_ids, attention_mask)

    # Convert to probabilities
    hate_probs = F.softmax(hate_logits, dim=1).cpu().numpy()[0]
    emotion_probs = F.softmax(emotion_logits, dim=1).cpu().numpy()[0]
    sentiment_probs = F.softmax(sentiment_logits, dim=1).cpu().numpy()[0]

    # -------------------------------
    # DEBUG CHECK
    # -------------------------------
    print("🔍 Emotion output shape:", emotion_probs.shape)

    if len(emotion_probs) != len(emotion_label_map):
        print("⚠️ WARNING: Model still uses OLD emotion classes (13)")
        print("👉 You need to retrain model with 5 classes\n")

    # -------------------------------
    # DISPLAY OUTPUT
    # -------------------------------

    print("\n📝 Input:", text)

    # -------------------------------
    # HATE
    # -------------------------------
    print("\n🚨 Hate Probabilities:")
    for i, prob in enumerate(hate_probs):
        print(f"{hate_label_map[i]}: {prob*100:.2f}%")

    final_hate = int(hate_probs.argmax())

    print("\n➡ FINAL HATE:", hate_label_map[final_hate])

    print("\n" + "-"*50)

    # -------------------------------
    # EMOTION
    # -------------------------------
    print("🧠 Top Emotions:")

    top2 = emotion_probs.argsort()[-2:][::-1]

    for idx in top2:
        idx = int(idx)

        if idx in emotion_label_map:
            label = emotion_label_map[idx]
        else:
            label = f"Unknown_{idx}"   # fallback for old model

        print(f"{label}: {emotion_probs[idx]*100:.2f}%")

    final_emotion = int(emotion_probs.argmax())

    if final_emotion in emotion_label_map:
        print("\n➡ FINAL EMOTION:", emotion_label_map[final_emotion])
    else:
        print("\n➡ FINAL EMOTION: Unknown")

    print("\n" + "-"*50)

    # -------------------------------
    # SENTIMENT
    # -------------------------------
    print("📊 Sentiment:")

    for i, prob in enumerate(sentiment_probs):
        print(f"{sentiment_label_map[i]}: {prob*100:.2f}%")

    final_sentiment = int(sentiment_probs.argmax())

    print("\n➡ FINAL SENTIMENT:", sentiment_label_map[final_sentiment])

    print("\n" + "="*60)

    # -------------------------------
    # RETURN STRUCTURED OUTPUT
    # -------------------------------
    return {
        "hate": hate_label_map[final_hate],
        "emotion": emotion_label_map.get(final_emotion, "Unknown"),
        "sentiment": sentiment_label_map[final_sentiment]
    }

In [123]:
telugu_predict("నీకు ఏమి తెలియదు 😡 నువ్వు పనికిరాని వాడివి")

🔍 Emotion output shape: (5,)

📝 Input: నీకు ఏమి తెలియదు 😡 నువ్వు పనికిరాని వాడివి

🚨 Hate Probabilities:
Hate: 26.61%
Offensive: 36.53%
Neutral: 36.86%

➡ FINAL HATE: Neutral

--------------------------------------------------
🧠 Top Emotions:
fear: 25.25%
sadness: 23.83%

➡ FINAL EMOTION: fear

--------------------------------------------------
📊 Sentiment:
Negative: 40.57%
Neutral: 29.46%
Positive: 29.97%

➡ FINAL SENTIMENT: Negative



{'hate': 'Neutral', 'emotion': 'fear', 'sentiment': 'Negative'}

# Model saving

In [124]:
save_path = "/content/drive/MyDrive/EmiHate/Models/indicbert_telugu"

# Save encoder + tokenizer
model.encoder.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Save full model weights (heads included)
import torch
torch.save(model.state_dict(), save_path + "/model_heads.pt")

print("✅ Telugu model saved locally")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Telugu model saved locally
